# Lab 12: Retrieval-Augmented Generation

**Business question:** Can a support answer or summary be traced to the right source passages?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | A made-up help center (70 passages) and three cached cases | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | 20 support questions, cached answers in four conditions, and the course complaint data | about 75–90 min (Exercise 2 means auditing ten claims against passages) | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

**Retrieval-augmented generation (RAG)** answers a question in two steps. First, a search finds the passages that should hold the answer. Then an LLM writes the answer from those passages and names its sources. You run the search yourself with the Module 6 tools. You do not call an LLM: I ran the prompts once and saved every answer in `data/llm_outputs`. A file of saved model answers like this is called a **cache**.

## Part 1 Guided core

### 1.1 Import the libraries

Select the `ban6403` kernel in VS Code, then run the cell below.

In [ ]:
# Path builds file paths that work on Windows, Mac and Linux; json reads the saved model answers.
from pathlib import Path
import json
import numpy as np
import pandas as pd
# TF-IDF for lexical search, as in Modules 4 and 6.
from sklearn.feature_extraction.text import TfidfVectorizer

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the course data is one folder up, in `data`. This lab reads from three of its folders: the help center in `rag`, the vectors in `embeddings` and the saved LLM answers in `llm_outputs`. Your saved files go to `outputs/m12`.

In [ ]:
# ".." means one folder up; the / operator joins folder and file names.
DATA = Path("../data")
RAG = DATA / "rag"
EMB = DATA / "embeddings"
LLM = DATA / "llm_outputs"
OUTPUT_DIR = Path("../outputs/m12")
# parents=True also creates the outputs folder if needed; exist_ok=True means no error if the folder exists.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 The help center

A support team answers questions from a **knowledge base**: help-center articles and policies. Our knowledge base belongs to **Fernhollow Financial, a made-up company**. Its fees, deadlines and rules were written for this course and do not describe any real bank.

Each article is split into short **passages**. Every passage has a stable **source ID**, such as `KB10-1` (article KB10, passage 1). An answer can cite these IDs, and a reviewer can look them up. The columns `effective_date` and `status` tell you when a policy started and whether it is still in force.

In [ ]:
articles = pd.read_csv(RAG / "m12_kb_articles.csv")
passages = pd.read_csv(RAG / "m12_kb_passages.csv")
print("articles:", len(articles), "| passages:", len(passages), "| words per passage:", passages["n_words"].min(), "to", passages["n_words"].max())
print(passages["status"].value_counts().to_string())
# Two passages side by side: the current fee schedule (KB10) and the old one (KB11).
passages.loc[passages["passage_id"].isin(["KB10-1", "KB11-1"]), ["passage_id", "title", "effective_date", "status", "text"]]

KB11 is an old fee schedule. It is kept for history and marked `archived`. KB10 replaced it. A good answer about today's fee must use KB10.

### 1.4 The support questions

I wrote 20 support questions. Some follow common themes in the course complaint data. For each question, the file lists the passages that answer it (`expected_source_ids`) and what a good answer should do (`expected_behavior`). Three questions cannot be answered from this help center, on purpose.

In [ ]:
# keep_default_na=False keeps empty cells as empty text (the unanswerable questions have no expected sources).
questions = pd.read_csv(RAG / "m12_questions.csv", keep_default_na=False)
# A lookup from question ID to question text: question_text["Q16"] gives the text of Q16.
question_text = questions.set_index("question_id")["question"]
print(questions["question_type"].value_counts().to_string())
questions.loc[questions["question_id"].isin(["Q12", "Q16", "Q17"]), ["question_id", "question", "question_type", "expected_source_ids", "expected_behavior"]]

### 1.5 Load the vectors

The passage and question vectors were made with the same model as in Modules 5 and 6 (all-MiniLM-L6-v2). Row 1 of each vector file belongs to row 1 of its table, and so on: one vector per passage and one per question.

In [ ]:
# One row per passage and one row per question, in the same order as the two tables.
passage_vectors = np.load(EMB / "m12_passages_minilm.npy").astype("float32")
question_vectors = np.load(EMB / "m12_questions_minilm.npy").astype("float32")
print("passage vectors:", passage_vectors.shape, "question vectors:", question_vectors.shape)

### 1.6 Search the help center

The helper `search` is the Module 6 retriever on a new collection. **Dense** search scores each passage by the cosine between its vector and the question vector. The saved vectors have length 1, so this cosine is a plain dot product. **Lexical** search uses TF-IDF with words and two-word phrases. Both search the column `search_text`, which is the article title plus the passage, so a short passage keeps its topic.

Question Q16 asks why the escrow account was charged for insurance.

In [ ]:
# Same TF-IDF setup as Module 6: words and two-word phrases (ngram_range=(1, 2)).
tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_passages = tfidf.fit_transform(passages["search_text"])
# A lookup from question ID to its row in question_vectors.
question_row = pd.Series(range(len(questions)), index=questions["question_id"])

def search(question_id, method="dense", k=10):
    """Take a question ID and "dense" or "lexical"; return the top k passages with rank and score."""
    if method == "lexical":
        # transform turns the question into a TF-IDF row with the vocabulary of the passages.
        query_tfidf = tfidf.transform([question_text[question_id]])
        # One cosine per passage; .ravel() turns the one-column result into a flat list.
        scores = (X_passages @ query_tfidf.T).toarray().ravel()
    else:
        # The saved vectors have length 1, so one dot product per passage is the cosine, as in Module 5.
        scores = passage_vectors @ question_vectors[question_row[question_id]]
    found = passages[["passage_id", "title", "status", "effective_date", "text"]].copy()
    found["score"] = scores
    # Sort from the highest score down; kind="stable" keeps tied scores in file order.
    found = found.sort_values("score", ascending=False, kind="stable").head(k)
    # Number the results 1, 2, 3, ... in a new first column.
    found.insert(0, "rank", range(1, len(found) + 1))
    return found.reset_index(drop=True)

print(question_text["Q16"])
# Four columns of the top 5, with scores rounded to 3 decimals.
search("Q16", k=5)[["rank", "passage_id", "title", "score"]].round(3)

### 1.7 Build a context pack

A **context pack** is the bounded set of passages that the LLM is allowed to use. Here it is the top k = 3 passages from dense search, in rank order. Each block starts with the source ID, the title, the effective date and the status, so the model can cite the ID and see how current the policy is.

The last line compares your pack with the input that I sent to the model. `True` means you rebuilt exactly the context the model saw, so every answer can be traced back to its passages.

In [ ]:
def pack_text(found):
    """Take a table of found passages; return the context pack as one text."""
    blocks = []
    # itertuples gives one row at a time; row.passage_id reads one column of that row.
    for row in found.itertuples():
        # A header line ([source ID] title | effective date | status), then the passage on the next line.
        blocks.append(f"[{row.passage_id}] {row.title} | effective {row.effective_date} | status: {row.status}\n{row.text}")
    # "\n\n" puts an empty line between the blocks.
    return "\n\n".join(blocks)

found_q16 = search("Q16", k=3)
pack_q16 = pack_text(found_q16)
print(pack_q16)
print()
print("source IDs:", found_q16["passage_id"].tolist(), "| words:", len(pack_q16.split()))
# The inputs I sent to the model for k = 3, one row per question.
inputs_k3 = pd.read_csv(LLM / "prompts/m12_input_grounded_k3.csv").set_index("question_id")
print("same as the cached input:", pack_q16 == inputs_k3.loc["Q16", "context"])

### 1.8 The grounded prompt

The **grounded** prompt tells the model to use only the passages and to list each claim with the source IDs that state it. When the passages do not answer the question, the model must answer "not enough information". The **ungrounded** prompt sends the question alone. Read both files in `data/llm_outputs/prompts`. The cell removes the prompt-version line, fills the grounded prompt with your pack and the question, and prints it in full.

In [ ]:
grounded_prompt = (LLM / "prompts/m12_grounded_v1_prompt.md").read_text()
# Line 1 holds the prompt version: split("\n", 1) cuts at the first line break, and [1] keeps the rest.
template = grounded_prompt.split("\n", 1)[1].strip()
# format fills the {context} and {question} slots of the template.
filled_q16 = template.format(context=pack_q16, question=question_text["Q16"])
print(grounded_prompt.splitlines()[0])
# The whole filled prompt: instructions, the three passages and the question.
print(filled_q16)
print("words in the filled prompt:", len(filled_q16.split()))

### 1.9 Load the cached answers

I sent all 20 questions to gemini-3.5-flash-lite in four conditions:

| Condition | Prompt | Context |
|---|---|---|
| `ungrounded` | `m12_ungrounded_v1` | none, the question only |
| `k1`, `k3`, `k8` | `m12_grounded_v1` | the top 1, 3 or 8 passages from dense search |

`k3` is the main condition. `k1` and `k8` change only the number of passages; this is a **top-k ablation** (change one setting, keep the rest).

All four runs used model gemini-3.5-flash-lite at temperature 0 on October 14, 2026, and no call failed. Each saved record also stores these details. The helper reads one file and joins each answer with its context IDs by `run_id`. The cell then prints the number of answers and the input tokens of each condition.

In [ ]:
def load_answers(condition):
    """Take a condition ("ungrounded", "k1", "k3" or "k8"); return its saved records and a table of its answers."""
    if condition == "ungrounded":
        name = "ungrounded"
    else:
        name = "grounded_" + condition
    # One JSON record per line.
    records = []
    for line in (LLM / f"m12_{name}_gemini.jsonl").read_text().splitlines():
        records.append(json.loads(line))

    rows = []
    for record in records:
        answer = record["parsed"]
        # Every source ID cited by any claim, without repeats, in sorted order.
        # A set keeps each value once: .add puts in an ID, and repeats are ignored.
        cited = set()
        for claim in answer["claims"]:
            for source_id in claim.get("source_ids", []):
                cited.add(source_id)
        rows.append({"run_id": record["id"],
                     "question_id": record["id"].split("_")[0],  # "Q16_k3" -> "Q16"
                     "condition": condition,
                     "status": answer["status"], "answer": answer["answer"], "claims": answer["claims"],
                     "cited": sorted(cited),
                     "input_tokens": record["input_tokens"], "latency_ms": record["latency_ms"]})
    out = pd.DataFrame(rows)

    # Add the IDs of the passages in the context pack: an empty list for ungrounded answers.
    if condition == "ungrounded":
        empty_lists = []
        for _ in range(len(out)):
            empty_lists.append([])
        out["context_ids"] = empty_lists
    else:
        inputs = pd.read_csv(LLM / f"prompts/m12_input_{name}.csv")
        out = out.merge(inputs[["run_id", "context_ids"]], on="run_id")
        # The file stores the IDs as text such as "KB13-1;KB13-2"; split(";") makes a list.
        out["context_ids"] = out["context_ids"].str.split(";")
    return records, out

caches = {}
for condition in ["ungrounded", "k1", "k3", "k8"]:
    caches[condition] = load_answers(condition)

# Rows and total input tokens per condition: more passages mean a longer prompt.
for condition in caches:
    records, out = caches[condition]
    print(f"{condition:>10} | rows {len(out)} | input tokens", out["input_tokens"].sum())

# One table with the answers of all four conditions; ignore_index numbers the rows again.
answer_tables = []
for records, out in caches.values():
    answer_tables.append(out)
answers = pd.concat(answer_tables, ignore_index=True)

### 1.10 Optional: try one context pack yourself (not graded)

You may paste the grounded prompt for question **Q19** (a wire sent to the wrong account) into Google AI Studio or UTSA Copilot Chat. The pack and question are made up for this course, so they are safe to paste.

1. Run the cell below and copy the printed prompt.
2. Paste it into the chat tool and read the answer.
3. Note which tool you used, whether it answered from the passages only, and whether it added outside knowledge or web links. A web link is not evidence from our help center.

If you skip this step or cannot get access, the second part of the cell is the captured equivalent: the cached answer from my run. Chat tools add their own instructions, so treat your trial as an illustration, not a measurement.

In [ ]:
pack_q19 = pack_text(search("Q19", k=3))
print(template.format(context=pack_q19, question=question_text["Q19"]))
print()
print("----- captured equivalent (cached Q19_k3) -----")
# The saved answer of run Q19_k3; .iloc[0] takes the one matching row.
q19 = answers[answers["run_id"].eq("Q19_k3")].iloc[0]
print(q19["status"], "|", q19["answer"])

### 1.11 Case 1: the same question with and without context

Compare the two answers to Q16. The ungrounded answer had only the question. The grounded answer had the k = 3 pack from step 1.7. Look at the status, the content and the citations.

In [ ]:
def show(run_id):
    """Take a run ID such as "Q16_k3"; print its status, context IDs, cited IDs and answer."""
    row = answers[answers["run_id"].eq(run_id)].iloc[0]
    print(f"{run_id} | status: {row['status']} | context: {row['context_ids']} | cited: {row['cited']}")
    print("   ", row["answer"])

print(question_text["Q16"])
show("Q16_ungrounded")
show("Q16_k3")

### 1.12 Case 2: a question the help center cannot answer

Q17 asks how to close the account of a customer who has died. No article covers this. Retrieval still returns three passages, because search always returns its top k. Watch what each answer does.

In [ ]:
print(question_text["Q17"])
# The answer without context, then the answer with the k = 3 pack.
show("Q17_ungrounded")
show("Q17_k3")

The ungrounded answer sounds helpful, but the help center says nothing about death certificates or executors. It may be good general advice, yet it is not Fernhollow's policy, and a customer may act on it. The grounded answer says "not enough information" and passes the question on. This is an **abstention**.

### 1.13 Audit claims against their sources

A fluent answer can still contain a claim that no source supports. A **claim audit** checks each claim on its own. The helper puts each claim next to the full text of the passages it cites, with their effective date and status. It also checks that each cited ID was in the context pack.

We use four verdicts, judged against the current help center:

| Verdict | Meaning |
|---|---|
| `supported` | a current passage states the claim, or several current passages together state it |
| `contradicted` | a current passage says something different (when two passages disagree, the newer effective date wins) |
| `unsupported` | no passage states the claim; this includes an inference that no passage states |
| `not_a_claim` | an apology, a bare referral to support, or a remark about the answer itself |

For grounded answers, `citation_ok` also asks whether the cited passage itself states the claim. It is blank for `not_a_claim`. The full rules are in `data/rag/README.md`.

In [ ]:
# Lookups from passage ID to its full text, its effective date and its status.
passage_text = passages.set_index("passage_id")["text"]
passage_date = passages.set_index("passage_id")["effective_date"]
passage_status = passages.set_index("passage_id")["status"]

def claim_table(run_id):
    """Take a run ID; return one row per claim with its cited IDs, whether they were in the pack, and their full text, date and status."""
    row = answers[answers["run_id"].eq(run_id)].iloc[0]
    rows = []
    # enumerate(..., 1) numbers the claims 1, 2, 3, ...
    for i, claim in enumerate(row["claims"], 1):
        cited = claim.get("source_ids", [])
        # Were all cited IDs in the context pack? Empty (None) when the claim cites nothing.
        if cited:
            cited_in_pack = True
            for source_id in cited:
                if source_id not in row["context_ids"]:
                    cited_in_pack = False
        else:
            cited_in_pack = None
        # Each cited passage that exists in the help center: its date and status, and its full text.
        cited_sources = []
        cited_texts = []
        for source_id in cited:
            if source_id in passage_text:
                cited_sources.append(f"{source_id} (effective {passage_date[source_id]}, {passage_status[source_id]})")
                cited_texts.append(passage_text[source_id])
        rows.append({"run_id": run_id, "claim_no": i, "claim": claim["claim"], "cited_ids": ";".join(cited),
                     "cited_in_pack": cited_in_pack, "cited_sources": "; ".join(cited_sources),
                     "cited_text": " | ".join(cited_texts)})
    return pd.DataFrame(rows)

def print_claims(table):
    """Take a claim table; print each claim with its cited passages in full."""
    for row in table.itertuples():
        print(f"claim {row.claim_no}: {row.claim}")
        print(f"   cited: {row.cited_sources} | in pack: {row.cited_in_pack}")
        print(f"   text: {row.cited_text}")

print_claims(claim_table("Q16_k3"))

Read each claim against its cited text. All three are stated in the cited passages, and every cited ID was in the pack.

The audit file holds a verdict for every claim in the four caches. These are draft judgments. Read the passage and compare your reasoning with the `note` column. A few rows are left blank for you in Practice. Here are the drafted verdicts for two of the answers above.

In [ ]:
# keep_default_na=False keeps the blank verdicts (yours to judge) as empty text.
audit = pd.read_csv(RAG / "m12_claim_audit_draft.csv", keep_default_na=False)
audit.loc[audit["run_id"].isin(["Q16_k3", "Q17_ungrounded"]), ["run_id", "claim_no", "claim", "verdict", "citation_ok", "note"]]

### 1.14 Case 3: retrieval failure or generation failure?

When a RAG answer is wrong or incomplete, ask where it broke:

- A **retrieval failure**: the passage that answers the question is not in the context pack. The model cannot use what it never saw.
- A **generation failure**: the right passage is in the pack, but the answer ignores it, adds an unsupported claim, or gives the wrong status.

Q12 asks what to do about a debt the customer does not recognize. The expected source is KB25-2, which explains the written dispute. Where did dense and lexical search rank it?

In [ ]:
print(question_text["Q12"])
expected_q12 = questions.set_index("question_id").loc["Q12", "expected_source_ids"]
for method in ["dense", "lexical"]:
    found = search("Q12", method)
    top3 = found["passage_id"].head(3).tolist()
    # The rank of the expected passage, as a list (empty when it is not in the top 10).
    expected_rank = found.loc[found["passage_id"].eq(expected_q12), "rank"].tolist()
    print(f"{method:>8} top 3: {top3} | rank of {expected_q12}:", expected_rank)
show("Q12_k3")

Dense search ranked KB25-2 seventh, so it was not in the k = 3 pack. The answer only describes the validation notice and never tells the customer how to dispute the debt. This is a **retrieval failure**. Notice too that the model still set its status to `answered`: the status flag did not warn us.

### 1.15 One top-k comparison

The k = 1 and k = 8 caches send the same prompt with fewer or more passages. Compare the three answers to Q12.

In [ ]:
# The Q12 answers of the three grounded conditions.
is_q12 = answers["question_id"].eq("Q12")
is_grounded = answers["condition"].isin(["k1", "k3", "k8"])
q12 = answers[is_q12 & is_grounded].copy()
# For each context pack (a list of IDs): is the expected passage in it?
in_pack = []
for ids in q12["context_ids"]:
    in_pack.append(expected_q12 in ids)
q12["expected_in_pack"] = in_pack
q12[["condition", "context_ids", "expected_in_pack", "status", "cited", "input_tokens"]]

In [ ]:
# The k = 8 answer to Q12: its pack now holds KB25-2.
show("Q12_k8")

With k = 8, KB25-2 is in the pack, and the answer explains the 30-day written dispute. More context fixed this retrieval failure, at the cost of a longer prompt (more input tokens). More passages can also add distractors; Practice looks at all 20 questions.

### 1.16 A first escalation rule

A support team needs a rule for when an answer goes to a person instead of the customer. This is an **abstention or escalation rule**. Here is a first version. It uses only what is known when the answer is made: the status, the citations and the context IDs.

In [ ]:
def needs_escalation(row):
    """Take one answer row; return True when the answer should go to a person."""
    if row["status"] != "answered":
        return True  # the model says it cannot fully answer
    if len(row["cited"]) == 0:
        return True  # no source for any claim
    # For two sets, a <= b means "every item of a is also in b".
    if not set(row["cited"]) <= set(row["context_ids"]):
        return True  # cites a passage it was not given
    return False

cases = answers[answers["run_id"].isin(["Q16_k3", "Q17_k3", "Q12_k3"])].copy()
# apply(..., axis=1) runs the function once for each row.
cases["escalate"] = cases.apply(needs_escalation, axis=1)
cases[["run_id", "status", "cited", "escalate"]]

The rule sends Q17 to a person and lets Q16 through, as it should. But it lets Q12 through, because the model said `answered` and cited a passage from its pack. You will revise this rule in Practice.

### 1.17 Save your outputs

This writes the Q16 context pack, the claim tables of the three cases and the rule results to `outputs/m12`. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
# The Q16 pack with its settings, source IDs and scores, saved as JSON.
pack_record = {"question_id": "Q16", "question": question_text["Q16"], "retriever": "dense, all-MiniLM-L6-v2", "k": 3,
               "source_ids": found_q16["passage_id"].tolist(), "scores": found_q16["score"].round(4).tolist(), "context": pack_q16}
(OUTPUT_DIR / "context_pack_Q16.json").write_text(json.dumps(pack_record, indent=2))

# The claim tables of four answers, stacked into one table.
case_tables = []
for run_id in ["Q16_k3", "Q16_ungrounded", "Q17_ungrounded", "Q12_k3"]:
    case_tables.append(claim_table(run_id))
pd.concat(case_tables).to_csv(OUTPUT_DIR / "case_claims.csv", index=False)
cases[["run_id", "status", "escalate"]].to_csv(OUTPUT_DIR / "case_escalation.csv", index=False)
print("saved in outputs/m12: context_pack_Q16.json, case_claims.csv, case_escalation.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. In Case 3 (Q12, k = 3), was the problem a retrieval failure or a generation failure? Use the pack IDs and the rank of the expected source.
2. The rule in step 1.16 did not escalate Q12_k3. Name one signal, available when the answer is made, that could have caught it.

**Your answer:**

## Part 2 Practice: 20 support questions

Now use all 20 questions and all four conditions. The questions and the help center are made up, but their themes come from real complaints, so we start there.

### 2.1 Which questions matter most?

Each question has a `complaint_theme`: the CFPB issue label it matches. Count how often each theme appears in the training complaints. A theme with many complaints is a question the support team will see often.

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
# Number of train complaints for each issue label.
train_issues = complaints.loc[complaints["split"].eq("train"), "issue"].value_counts()
themes = questions[["question_id", "question_type", "complaint_theme"]].copy()
# map looks up each theme in train_issues; a theme with no complaints gets NaN, which fillna(0) makes 0.
themes["train_complaints"] = themes["complaint_theme"].map(train_issues).fillna(0).astype(int)
themes.sort_values("train_complaints", ascending=False).head(8)

### 2.2 Did retrieval find the expected sources?

For the 17 questions with expected sources, count how many of them are in the top k. Compare dense search at k = 1, 3 and 8, and lexical search at k = 3. `all_found` counts questions with every expected source in the top k; `any_found` counts questions with at least one.

This is **recall at k**: for one question, the share of its needed passages that appear in the top k. `all_found` counts questions with recall at k of 1; `any_found` counts questions with recall above 0. Module 6 measured **precision at 5** (the share of the top 5 that is relevant) and **reciprocal rank** (1 divided by the rank of the first relevant result). For a context pack the question is different: did the evidence get into the pack at all?

In [ ]:
# The expected source IDs of each answerable question, as a list (questions without sources are skipped).
expected = {}
for qid, sources in zip(questions["question_id"], questions["expected_source_ids"]):
    if sources:
        expected[qid] = sources.split(";")

def retrieval_check(method, k):
    """Take a method and k; return how many questions have all, or any, of their expected sources in the top k."""
    rows = []
    for qid, sources in expected.items():
        top = search(qid, method, k)["passage_id"].tolist()
        n_found = 0
        for source_id in sources:
            if source_id in top:
                n_found += 1
        rows.append({"question_id": qid, "found": n_found, "needed": len(sources)})
    results = pd.DataFrame(rows)
    return {"method": method, "k": k,
            "all_found": int((results["found"] == results["needed"]).sum()),
            "any_found": int((results["found"] > 0).sum()),
            "questions": len(results)}

checks = []
for method, k in [("dense", 1), ("dense", 3), ("dense", 8), ("lexical", 3)]:
    checks.append(retrieval_check(method, k))
pd.DataFrame(checks)

### 2.3 What did each condition answer?

Count the status of the answers in each condition. Then look at the three unanswerable questions.

In [ ]:
print(pd.crosstab(answers["condition"], answers["status"]))
unanswerable = questions.loc[questions["question_type"].eq("unanswerable"), "question_id"]
# pivot: one row per question, one column per condition, the status in the cells.
answers[answers["question_id"].isin(unanswerable)].pivot(index="question_id", columns="condition", values="status")

### 2.4 Drafted claim verdicts by condition

Join every claim with its drafted verdict and count verdicts by condition. Rows marked `student` are yours to judge in Exercise 2; they are counted as `to_judge`.

In [ ]:
# The empty verdicts (the rows for you to judge) become "to_judge".
verdicts = audit["verdict"].replace("", "to_judge")
pd.crosstab(audit["condition"], verdicts)

### 2.5 Retrieval failure or generation failure, for every k = 3 answer

The cell labels each k = 3 answer with one diagnosis, checked in this order:

1. **correct abstention**: an unanswerable question, and the model said "not enough information";
2. **should abstain**: an unanswerable question that got an answer anyway;
3. **retrieval failure**: no expected source in the pack;
4. **stale source**: a claim is contradicted by a newer passage, and a person checked that the claim repeats an older passage that was in the pack (the list `STALE_CHECKED`);
5. **generation failure**: an unsupported or contradicted claim when the needed evidence was in the pack, or "not enough information" although every expected source was in the pack;
6. **partial retrieval**: some expected sources are missing from the pack;
7. **ok**: everything else.

A contradiction alone does not show that a source was out of date. The model can also invent a number, such as "60 days" when the current passage says "30 days". That is why the stale-source label needs a person's check.

This uses the expected sources and the drafted verdicts, so it is an evaluation tool. A live system would not know them. Claims without a verdict count as fine, so the cell also lists the answers whose audit is incomplete.

In [ ]:
# For each run ID, the list of its unsupported or contradicted verdicts.
bad_claims = audit[audit["verdict"].isin(["unsupported", "contradicted"])]
bad = bad_claims.groupby("run_id")["verdict"].apply(list)
# Runs that still have claims without a verdict (yours to judge in Exercise 2).
unjudged = audit.loc[audit["verdict"].eq(""), "run_id"].unique().tolist()
# Runs whose contradicted claim I checked by reading: it repeats an older passage from the pack
# (KB03-1, 2021: 60 days) that a newer current passage (KB01-2, 2026: 120 days) replaces.
STALE_CHECKED = ["Q01_k3", "Q01_k8"]

def diagnose(row):
    """Take one answer row; return its diagnosis: the first rule in the list above that applies."""
    sources = expected.get(row["question_id"], [])  # an empty list for an unanswerable question
    # How many expected sources are in the context pack.
    in_pack = 0
    for source_id in sources:
        if source_id in row["context_ids"]:
            in_pack += 1
    problems = bad.get(row["run_id"], [])
    if not sources:
        if row["status"] == "not_enough_information":
            return "correct abstention"
        return "should abstain"
    if in_pack == 0:
        return "retrieval failure"
    if "contradicted" in problems and row["run_id"] in STALE_CHECKED:
        return "stale source"
    if problems or (row["status"] == "not_enough_information" and in_pack == len(sources)):
        return "generation failure"
    if in_pack < len(sources):
        return "partial retrieval"
    return "ok"

k3 = answers[answers["condition"].eq("k3")].copy()
k3["diagnosis"] = k3.apply(diagnose, axis=1)
print(k3["diagnosis"].value_counts().to_string())
# A diagnosis can change once the missing verdicts are filled in.
k3_unjudged = []
for run_id in k3["run_id"]:
    if run_id in unjudged:
        k3_unjudged.append(run_id)
print("audit incomplete (claims still to judge):", k3_unjudged)
# ~ means "not": show the answers whose diagnosis is neither ok nor a correct abstention.
k3.loc[~k3["diagnosis"].isin(["ok", "correct abstention"]), ["question_id", "status", "context_ids", "cited", "diagnosis"]]

### 2.6 The top-k ablation for all 20 questions

Apply the same diagnosis to k = 1 and k = 8, and add the cost: the median input tokens and the median time per call.

In [ ]:
rows = []
for condition in ["k1", "k3", "k8"]:
    out = answers[answers["condition"].eq(condition)].copy()
    # Count the diagnoses in this condition, then add the median cost.
    diagnosis_counts = out.apply(diagnose, axis=1).value_counts()
    row = {"condition": condition}
    row.update(diagnosis_counts.to_dict())
    row["median_input_tokens"] = int(out["input_tokens"].median())
    row["median_latency_ms"] = int(out["latency_ms"].median())
    rows.append(row)
cols = ["ok", "partial retrieval", "retrieval failure", "stale source", "generation failure", "correct abstention", "should abstain", "median_input_tokens", "median_latency_ms"]
# reindex puts the columns in this order and adds a missing one as NaN; fillna(0) makes it 0.
pd.DataFrame(rows).set_index("condition").reindex(columns=cols).fillna(0).astype(int)

### Exercise 1: build and bound a context pack

A bounded pack has a clear limit on what goes in. Use question **Q05** ("How much is the overdraft fee now?") and question **Q01** (the dispute deadline).

1. For each question, build the k = 3 dense pack with `search` and show `passage_id`, `status` and `effective_date`.
2. Build a second pack that keeps only passages with `status == "current"` before taking the top 3. Hint: `search(qid, k=10)`, filter the rows, then `.head(3)`.
3. Save both Q05 packs (IDs and text from `pack_text`) to `outputs/m12/context_pack_Q05.json`.
4. Print which IDs the filter removed for each question. Does the filter remove the outdated source in Q01? Look at KB03's dates.

In [ ]:
# Exercise 1: your code here
# TODO: for Q05 and Q01, build the k = 3 dense pack with search and show passage_id, status and effective_date.
# TODO: build a second pack from search(qid, k=10): keep status == "current", then .head(3).
# TODO: save both Q05 packs (IDs and pack_text) to OUTPUT_DIR / "context_pack_Q05.json".
# TODO: print the IDs the filter removed for each question, and check KB03's dates for Q01.

### Exercise 2: audit claims, grounded and ungrounded

Ten claims in the audit file are left for you (`judged_by == "student"`). They come from the k = 3 and ungrounded answers to **Q04** (a scam payment) and from the ungrounded answer to **Q15** (an escrow surplus).

1. Show these rows with `claim_table` (for the cited text), and print the passages you need: KB07-1, KB07-2, KB08-2, KB28-2 and KB29-2.
2. Give each claim a verdict (`supported`, `unsupported`, `contradicted` or `not_a_claim`) using the rule in step 1.13. For the Q04_k3 claims, also set `citation_ok` to yes or no. Write each reason from the passage in a comment. Fill the verdicts into `audit` with a True/False mask and `.loc`:

```python
# One entry per claim: (run ID, claim number) -> verdict.
my_verdicts = {("Q04_k3", 1): "...", ("Q04_k3", 2): "..."}   # TODO: all ten claims
for (run_id, claim_no), verdict in my_verdicts.items():
    # The mask is True only on the row of this claim.
    mask = audit["run_id"].eq(run_id) & audit["claim_no"].eq(claim_no)
    audit.loc[mask, "verdict"] = verdict
```

Do the same with a second dictionary for `citation_ok`.
3. Recount the verdicts by condition, as in step 2.4, for `k3` and `ungrounded` only.
4. Save the completed audit as `outputs/m12/claim_audit.csv`.
5. Rerun steps 2.4 to 2.6 (click in each cell and press Shift+Enter), so the diagnoses use your verdicts. Exercise 3 uses them.

There is no `partly supported` verdict. Some claims are close to a passage but drop a condition or overstate it. Ask whether the claim, as written, would mislead a customer. If not, mark it `supported`; if it would, mark it `unsupported`, or `contradicted` when a passage says something different. Add a comment with your reason from the passage. Reasoned alternatives are accepted.

In [ ]:
# Exercise 2: your code here
# TODO: show the claims with judged_by == "student" with claim_table, and print KB07-1, KB07-2, KB08-2, KB28-2, KB29-2.
# TODO: write my_verdicts = {("Q04_k3", 1): "...", ...}; for each claim, build a mask on run_id and claim_no
#       and set audit.loc[mask, "verdict"] (and citation_ok for Q04_k3). Write each reason in a comment.
# TODO: recount the verdicts by condition for k3 and ungrounded, as in step 2.4.
# TODO: save the completed audit as OUTPUT_DIR / "claim_audit.csv".
# TODO: then rerun the cells of steps 2.4 to 2.6.

### Exercise 3: revise the escalation rule

The rule in step 1.16 lets some answers through that a person should check. Write `needs_escalation_v2(row)` using only signals available when the answer is made. Ideas: the status, the citations, the dense score of the top passage, or how many claims there are. You could also check whether the pack holds an archived passage, or two passages on one topic with different dates.

1. Apply both rules to the 20 k = 3 answers.
2. Make a table with `question_id`, `diagnosis` (from step 2.5), `rule_v1` and `rule_v2`.
3. Count, for each rule, how many answers it escalates, and how many problem answers it lets through. A problem answer has any diagnosis except the two successes, `ok` and `correct abstention`. Count separately the correct abstentions that the rule does not escalate.
4. Save the table as `outputs/m12/escalation_k3.csv`.

**Be honest about the counts.** You design and check the rule on the same 20 answers, so the counts flatter it. Module 9's rule was "choose on validation, test once". A fair count needs new questions the rule has not seen, as Module 13 does with held-out complaints.

In [ ]:
# Exercise 3: your code here
# TODO: write needs_escalation_v2(row) with signals known when the answer is made.
# TODO: apply both rules to the k3 answers; make a table with question_id, diagnosis, rule_v1 and rule_v2.
# TODO: count, for each rule, the answers it escalates and the problem answers it lets through
#       (problem = any diagnosis except "ok" and "correct abstention"); count the abstentions it does not escalate.
# TODO: save the table as OUTPUT_DIR / "escalation_k3.csv".

### Practice interpretation

Write 3–5 sentences:

- Using steps 2.3 and 2.4 and Exercise 2, how did grounding change the answers? Give counts of unsupported or contradicted claims for `ungrounded` and `k3`.
- Using steps 2.2, 2.5 and 2.6, were the remaining k = 3 problems mostly retrieval failures or generation failures? What did k = 1 and k = 8 change, and at what token cost?
- State your escalation rule from Exercise 3 in one or two plain sentences. How many answers does it send to a person, and which problem does it still miss? Why might it do worse on new questions?

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

Did you use an AI tool to explain or debug code, or for the optional trial in step 1.10? Then add one line here. Name the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Hybrid retrieval for the context pack

Module 6 combined lexical and dense lists with reciprocal rank fusion (RRF): each passage gets `1 / (60 + rank)` from each list. Build hybrid k = 3 packs and repeat the `all_found` and `any_found` counts from step 2.2. Does hybrid search put KB25-2 into the Q12 pack, or KB07-1 into the Q04 pack? There are no cached answers for hybrid packs, so you could try one in Google AI Studio.

In [ ]:
# Challenge A: your code here
# TODO: for each question, combine the dense and lexical top lists with RRF: 1 / (60 + rank) from each list.
# TODO: take the hybrid top 3 and repeat the all_found and any_found counts of step 2.2.

### B. A quick automatic support check

Can a computer flag unsupported claims? One cheap signal is the TF-IDF cosine between a claim and its cited passage. Compute it for every grounded claim with a drafted verdict, and compare the scores of `supported` and `contradicted` claims. What would this check miss?

In [ ]:
# Challenge B: your code here
# TODO: for each grounded claim with a drafted verdict, compute the TF-IDF cosine between the claim and its cited passage.
# TODO: compare the scores of supported and contradicted claims.

### C. Beyond one help center

Production systems go further. **Agents** decide which tool or search to call next. **Knowledge graphs** store facts as linked entities. Pipelines for **long documents** split and summarize them in stages. **GraphRAG** retrieves from a graph built over the documents. Each adds steps that can fail. For one of them, write which new failure type it adds to the two in this lab, and how you would audit it.

## Further reading

- [Jurafsky and Martin, Speech and Language Processing, chapter on question answering, information retrieval and RAG](https://web.stanford.edu/~jurafsky/slp3/)
- [Lewis et al. (2020), Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [Liu et al. (2023), Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172)

Next, Module 13 evaluates an LLM workflow on held-out cases and asks what evidence would justify using it in a business process.